## Predicting Heart Diseases:

### Section Map (Aligned with Notebook)
1) ## **Load Data**
- Load train/test datasets and verify basic shapes.

2) ## **Check Data Quality**
- Review missing values, dtypes, and basic distributions.

3) ## **Stratified K-Folds for Model Comparison**
- Compare models with class-balanced CV splits.
- Current comparison focus is ROC-AUC.

4) ## **HyperParameter Tuning**
- Run Optuna tuning for LightGBM.
- Use fold validation as `eval_set` with early stopping.

5) ## **Retrain Best Model with Best HyperParameters**
- Retrain with selected best params and check hold-out performance.

6) ## **Evaluate Feature Importance**
- Inspect global signal using Permutation Importance.

7) ## **SHAP-based Feature Attribution**
- Add SHAP bar and beeswarm plots for interpretation.

8) ## **Get Predictions and Store in "submission.csv"**
- Build the baseline single-model submission file.

9) ## **Optuna Tuning for XGBoost and CatBoost**
- Extend Optuna tuning to XGBoost and CatBoost.
- CatBoost uses selected categorical columns and GPU settings.
- Optuna objective uses rank-based scoring.

10) ## **Ensemble: Rank-Based Weighted Averaging**
- Compute model weights from OOF rank scores.
- Average fold test predictions and combine with rank averaging to create `submission_ensemble.csv`.



In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.metrics import classification_report, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
import xgboost as xgb
import catboost as ctb

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import optuna

from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings("ignore")

sns.set_theme()


## Load Data

In [ ]:
class Config:
    train_path = "/kaggle/input/playground-series-s6e2/train.csv"
    test_path = "/kaggle/input/playground-series-s6e2/test.csv"

In [ ]:
train_data = pd.read_csv(Config.train_path)
test_data = pd.read_csv(Config.test_path)

train_data.head()

In [ ]:
print("Total Train Samples:", len(train_data))
print("Total Test Samples:", len(test_data))

## Check Data Quality

In [ ]:
# Check for Missing Data
print("Null Counts for Each Column:\n")
display(train_data.isna().sum())

print("Data Type for Each Column:\n")
display(train_data.info())

In [ ]:
# Plot Distribution for Each Feature Column
target = "Heart Disease"
features = [i for i in train_data.columns[1:] if i!=target]

for i in features:
    train_data[i].hist()
    plt.title(f"Distribution of {i}")
    plt.xlabel("Value")
    plt.ylabel("Counts")
    plt.show()

In [ ]:
# Plot Distribution for Each Target Column
train_data[target].hist()
plt.title(f"Distribution of {target}")
plt.xlabel("Value")
plt.ylabel("Counts")
plt.show()

## Stratified K-Folds for Model Comparison

In [ ]:
def evaluate_model(ModelClass, X, y, splits=4, verbose=False, **kwargs):
    skf = StratifiedKFold(n_splits=splits)
    auc_scores = []

    scale_model_names = {
        'LogisticRegression',
        'GaussianNB',
        'SVC',
        'KNeighborsClassifier'
    }

    for i, (train_index, test_index) in enumerate(skf.split(X, y)):
        if verbose:
            print(f"Fold {i}:")
            print("Train Samples:", len(train_index))
            print("Test Samples:", len(test_index))

        X_train_raw = train_data.loc[train_index, features].copy()
        X_test_raw = train_data.loc[test_index, features].copy()
        y_train = y[train_index]
        y_test = y[test_index]

        if ModelClass.__name__ in scale_model_names:
            scaler = StandardScaler().set_output(transform="pandas")
            X_train = scaler.fit_transform(X_train_raw)
            X_test = scaler.transform(X_test_raw)
        else:
            X_train = X_train_raw
            X_test = X_test_raw

        # For CatBoost, keep categorical features as numeric categories (Int64).
        if ModelClass.__name__ == 'CatBoostClassifier' and 'cat_features' in kwargs:
            for col in kwargs['cat_features']:
                if col in X_train.columns:
                    X_train[col] = pd.to_numeric(X_train[col], errors='coerce').fillna(-1).astype('Int64')
                    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(-1).astype('Int64')

        model = ModelClass(**kwargs)
        model.fit(X_train, y_train)

        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)[:, 1]
        else:
            y_prob = model.decision_function(X_test)

        score = roc_auc_score(y_test, y_prob)
        auc_scores.append(score)

        if verbose:
            print("ROC-AUC:", score)

    return np.mean(auc_scores), np.std(auc_scores)



In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train_data[target])

print('Total Classes:', len(label_encoder.classes_))
print("Classes:", label_encoder.classes_)
CAT_FEATURES = [
    'Sex',
    'Chest pain type',
    'FBS over 120',
    'EKG results',
    'Slope of ST',
    'Thallium',
    'Number of vessels fluro',
    'Exercise angina'
]

exps = [
    {"ModelClass": LogisticRegression, "X": train_data, "y": y, "class_weight":"balanced"}, 
    {"ModelClass": GaussianNB, "X": train_data, "y": y},
    # {"ModelClass": RandomForestClassifier, "X": train_data, "y": y, "class_weight":"balanced"},
    # {"ModelClass": DecisionTreeClassifier, "X": train_data, "y": y, "class_weight":"balanced"},
    # {"ModelClass": SVC, "X": train_data, "y": y, "class_weight":"balanced"},
    # {"ModelClass": KNeighborsClassifier, "X": train_data, "y": y},
    # {"ModelClass": GradientBoostingClassifier, "X": train_data, "y": y},
    {"ModelClass": ctb.CatBoostClassifier, "X": train_data, "y": y, "verbose": False, "logging_level":"Silent", "task_type":"GPU", "devices":"0", "eval_metric":"AUC", "cat_features":CAT_FEATURES},
    {"ModelClass": lgb.LGBMClassifier, "X": train_data, "y": y, "class_weight":"balanced", "verbosity": 0}, 
    {"ModelClass": xgb.XGBClassifier, "X": train_data, "y": y, "class_weight":"balanced", "verbosity": 0},
]

for param in exps: 
    mean, std = evaluate_model(**param)
    print(f"Model: {param['ModelClass'].__name__}\nROC-AUC Mean: {mean} | ROC-AUC STD: {std}")



## HyperParameter Tuning

In [ ]:
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

def rank_auc_score(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    ranks = pd.Series(y_prob).rank(method='average').to_numpy()
    n_pos = np.sum(y_true == 1)
    n_neg = np.sum(y_true == 0)
    if n_pos == 0 or n_neg == 0:
        return 0.5
    sum_ranks_pos = ranks[y_true == 1].sum()
    return float((sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))

def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "n_jobs": -1,
        "class_weight": "balanced",
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 512),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 75, 125),
    }

    rank_scores = []

    for train_idx, val_idx in skf.split(train_data[features], y):
        X_train_fold, X_val_fold = train_data[features].iloc[train_idx], train_data[features].iloc[val_idx]
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]

        scaler = StandardScaler().set_output(transform="pandas")
        X_train_fold = scaler.fit_transform(X_train_fold)
        X_val_fold = scaler.transform(X_val_fold)

        model = lgb.LGBMClassifier(**params)

        model.fit(
            X_train_fold,
            y_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(100, verbose=False),
                lgb.log_evaluation(0)
            ]
        )

        pred_proba = model.predict_proba(X_val_fold)[:, 1]
        rank_score = rank_auc_score(y_val_fold, pred_proba)
        rank_scores.append(rank_score)

        trial.report(np.mean(rank_scores), step=len(rank_scores))
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(rank_scores)



In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner()
)

study.optimize(objective, n_trials=100)

print("Best Rank Score:", study.best_value)
print("Best Params:", study.best_params)



## Retrain Best Model with Best HyperParameters

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(train_data[features], y, test_size=0.1, random_state=42, stratify=y)

scaler = StandardScaler().set_output(transform="pandas")
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = lgb.LGBMClassifier(class_weight="balanced", verbosity=0, **study.best_params)
model.fit(X_train, y_train)

In [ ]:
f1_score(y_test, model.predict(X_test))

## Evaluate Feature Importance

In [ ]:
# Permutation Importance
perm_importance = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)
perm_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance Mean': perm_importance.importances_mean,
    'Importance Std': perm_importance.importances_std
})
print("\nPermutation Importance:")
print(perm_importance_df.sort_values(by='Importance Mean', ascending=False))

## SHAP-based Feature Attribution

Add SHAP explanations for the trained LightGBM model and visualize global importance.

In [ ]:
# SHAP explanation for the trained model
try:
    import shap
except ImportError:
    print("SHAP is not installed. Install with: pip install shap")
else:
    # Use a sample for faster SHAP computation
    if len(X_test) > 5000:
        X_shap = X_test.sample(5000, random_state=42)
    else:
        X_shap = X_test.copy()

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)

    # SHAP output shape can vary by SHAP version
    if isinstance(shap_values, list):
        shap_plot_values = shap_values[1]
    else:
        shap_plot_values = shap_values

    print(f"Computed SHAP on {len(X_shap)} samples")

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_plot_values, X_shap, plot_type='bar', show=False)
    plt.title("SHAP Global Importance (Bar)")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_plot_values, X_shap, show=False)
    plt.title("SHAP Summary (Beeswarm)")
    plt.tight_layout()
    plt.show()



## Get Predictions and Store in "submission.csv"

In [ ]:
print(label_encoder.classes_)

In [ ]:
test_df = pd.read_csv(Config.test_path)
X_test = test_df[features]
X_test = scaler.transform(X_test)
y_prob = model.predict_proba(X_test)
submission = pd.DataFrame({
    "id": test_df["id"], 
    "Heart Disease": y_prob[:, 1]
})
submission.to_csv("submission.csv", index=False)

In [ ]:
submission.head()

## Optuna Tuning for XGBoost and CatBoost

Tune XGBoost/CatBoost with Stratified K-Fold rank-based validation before ensemble scoring.


In [ ]:
# Optuna tuning for XGBoost and CatBoost (run before ensemble)
X_opt = train_data[features].reset_index(drop=True)
y_opt = np.asarray(y)
skf_opt = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

# Lower this if runtime is long
N_TRIALS_XGB = 20
N_TRIALS_CAT = 20

CAT_FEATURES = [
    'Sex',
    'Chest pain type',
    'FBS over 120',
    'EKG results',
    'Slope of ST',
    'Thallium',
    'Number of vessels fluro',
    'Exercise angina'
]

def rank_auc_score(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    ranks = pd.Series(y_prob).rank(method='average').to_numpy()
    n_pos = np.sum(y_true == 1)
    n_neg = np.sum(y_true == 0)
    if n_pos == 0 or n_neg == 0:
        return 0.5
    sum_ranks_pos = ranks[y_true == 1].sum()
    return float((sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))

def fit_with_valid_ranking_early_stopping(model_name, model, X_tr, y_tr, X_va, y_va):
    if model_name == 'xgb':
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )
    elif model_name == 'catboost':
        model.fit(
            X_tr,
            y_tr,
            eval_set=(X_va, y_va),
            cat_features=CAT_FEATURES,
            use_best_model=True,
            verbose=False
        )
    else:
        model.fit(X_tr, y_tr)

def cv_mean_rank(model_name, make_model):
    scores = []
    for tr_idx, va_idx in skf_opt.split(X_opt, y_opt):
        X_tr, X_va = X_opt.iloc[tr_idx], X_opt.iloc[va_idx]
        y_tr, y_va = y_opt[tr_idx], y_opt[va_idx]

        model = make_model()
        fit_with_valid_ranking_early_stopping(model_name, model, X_tr, y_tr, X_va, y_va)
        pred_proba = model.predict_proba(X_va)[:, 1]
        scores.append(rank_auc_score(y_va, pred_proba))
    return float(np.mean(scores))

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1200),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 12),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'random_state': 42,
        'verbosity': 0,
        'n_jobs': -1,
        'early_stopping_rounds': 50
    }
    return cv_mean_rank('xgb', lambda: xgb.XGBClassifier(**params))

def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 1200),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 20.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 10.0),
        'loss_function': 'Logloss',
        'eval_metric': 'Logloss',
        'custom_metric': ['AUC'],
        'random_seed': 42,
        'verbose': False,
        'allow_writing_files': False,
        'task_type': 'GPU',
        'devices': '0',
        'od_type': 'Iter',
        'od_wait': 50
    }
    return cv_mean_rank('catboost', lambda: ctb.CatBoostClassifier(**params))

xgb_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner()
)
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS_XGB, show_progress_bar=True)

cat_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner()
)
cat_study.optimize(cat_objective, n_trials=N_TRIALS_CAT, show_progress_bar=True)

xgb_best_params = {
    **xgb_study.best_params,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'random_state': 42,
    'verbosity': 0,
    'n_jobs': -1,
    'early_stopping_rounds': 50
}

catboost_best_params = {
    **cat_study.best_params,
    'loss_function': 'Logloss',
    'eval_metric': 'Logloss',
    'custom_metric': ['AUC'],
    'random_seed': 42,
    'verbose': False,
    'allow_writing_files': False,
    'task_type': 'GPU',
    'devices': '0',
    'od_type': 'Iter',
    'od_wait': 50
}

print('XGBoost Best Rank Score:', xgb_study.best_value)
print('XGBoost Best Params:', xgb_best_params)
print('CatBoost Best Rank Score:', cat_study.best_value)
print('CatBoost Best Params:', catboost_best_params)



## Ensemble: Rank-Based Weighted Averaging

Evaluate models with OOF rank score and build a rank-averaged ensemble submission.


In [ ]:
# Ensemble with OOF rank-based evaluation + weighted rank averaging

from sklearn.model_selection import StratifiedKFold

X_full = train_data[features].reset_index(drop=True)
y_full = np.asarray(y)

# Reuse test data if already loaded in previous cells
if 'test_df' not in globals():
    test_df = pd.read_csv(Config.test_path)
X_test_full = test_df[features]

print('Label classes:', list(label_encoder.classes_))

CAT_FEATURES = [
    'Sex',
    'Chest pain type',
    'FBS over 120',
    'EKG results',
    'Slope of ST',
    'Thallium',
    'Number of vessels fluro',
    'Exercise angina'
]

def rank_auc_score(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    ranks = pd.Series(y_prob).rank(method='average').to_numpy()
    n_pos = np.sum(y_true == 1)
    n_neg = np.sum(y_true == 0)
    if n_pos == 0 or n_neg == 0:
        return 0.5
    sum_ranks_pos = ranks[y_true == 1].sum()
    return float((sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))

def percentile_rank(values):
    return pd.Series(values).rank(method='average', pct=True).to_numpy()

xgb_params_for_ensemble = globals().get('xgb_best_params', {
    'n_estimators': 500,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'reg_lambda': 1.0,
    'reg_alpha': 0.0,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'random_state': 42,
    'verbosity': 0,
    'n_jobs': -1,
    'early_stopping_rounds': 50
})

catboost_params_for_ensemble = globals().get('catboost_best_params', {
    'iterations': 500,
    'depth': 6,
    'learning_rate': 0.05,
    'loss_function': 'Logloss',
    'eval_metric': 'Logloss',
    'custom_metric': ['AUC'],
    'random_seed': 42,
    'verbose': False,
    'allow_writing_files': False,
    'task_type': 'GPU',
    'devices': '0',
    'od_type': 'Iter',
    'od_wait': 50
})

model_factories = {
    'lgbm': lambda: lgb.LGBMClassifier(class_weight='balanced', verbosity=0, **study.best_params),
    'xgb': lambda: xgb.XGBClassifier(**xgb_params_for_ensemble),
    'catboost': lambda: ctb.CatBoostClassifier(**catboost_params_for_ensemble)
}

skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

def fit_with_valid_ranking_early_stopping(model_name, model, X_tr, y_tr, X_va, y_va):
    if model_name == 'lgbm':
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric='auc',
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]
        )
    elif model_name == 'xgb':
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    elif model_name == 'catboost':
        model.fit(
            X_tr,
            y_tr,
            eval_set=(X_va, y_va),
            cat_features=CAT_FEATURES,
            use_best_model=True,
            verbose=False
        )
    else:
        model.fit(X_tr, y_tr)

oof_probs = {}
model_rank = {}
test_probs = {}

for model_name, make_model in model_factories.items():
    fold_probs = np.zeros(len(X_full), dtype=float)
    fold_test_probs = []

    for _, (tr_idx, va_idx) in enumerate(skf.split(X_full, y_full), start=1):
        X_tr = X_full.iloc[tr_idx]
        y_tr = y_full[tr_idx]
        X_va = X_full.iloc[va_idx]
        y_va = y_full[va_idx]

        model = make_model()
        fit_with_valid_ranking_early_stopping(model_name, model, X_tr, y_tr, X_va, y_va)

        fold_probs[va_idx] = model.predict_proba(X_va)[:, 1]
        fold_test_probs.append(model.predict_proba(X_test_full)[:, 1])

    test_probs[model_name] = np.mean(fold_test_probs, axis=0)
    score = rank_auc_score(y_full, fold_probs)
    oof_probs[model_name] = fold_probs
    model_rank[model_name] = score
    print(f"{model_name} OOF Rank Score={score:.6f}")

rank_margin = {name: max(score - 0.5, 1e-6) for name, score in model_rank.items()}
margin_sum = sum(rank_margin.values())
weights = {name: margin / margin_sum for name, margin in rank_margin.items()}
print('Ensemble Weights (Rank margin normalized):', weights)

ensemble_oof = np.zeros(len(X_full), dtype=float)
for name in model_factories:
    ensemble_oof += weights[name] * percentile_rank(oof_probs[name])

ensemble_rank = rank_auc_score(y_full, ensemble_oof)
print(f"Ensemble OOF Rank Score={ensemble_rank:.6f}")

ensemble_test_prob = np.zeros(len(X_test_full), dtype=float)
for name in model_factories:
    ensemble_test_prob += weights[name] * percentile_rank(test_probs[name])

submission_ensemble = pd.DataFrame({
    'id': test_df['id'],
    'Heart Disease': ensemble_test_prob
})
submission_ensemble.to_csv('submission_ensemble.csv', index=False)

submission_ensemble.head()

